In [1]:
import gensim
from gensim.models import KeyedVectors

amazon_file_path = 'webzio_amazon.json'
tesla_file_path = "webzio_tesla.json"
model_file_path = "GoogleNews-vectors-negative300.bin"

In [2]:
model = KeyedVectors.load_word2vec_format(model_file_path, binary=True)

Exploring the dataset for one line to see the structure of the json dataset

In [3]:
import json

with open(amazon_file_path, 'r', encoding='utf-8') as f:
    first_record = json.loads(f.readline())

print("Title:", first_record.get("title"))
print("\nAvailable keys:", list(first_record.keys()))

Title: Amazon Singapore Celebrates Prime Day with Prime Day Carnival

Available keys: ['thread', 'uuid', 'url', 'ord_in_thread', 'parent_url', 'author', 'published', 'title', 'text', 'highlightText', 'highlightTitle', 'highlightThreadTitle', 'language', 'external_links', 'external_images', 'entities', 'rating', 'crawled', 'updated']


In [4]:
def read_json(file_path):
    titles = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                record = json.loads(line)
                title = record.get('title')
                if title and title.strip():
                    titles.append(title)
    return titles

# DATA Exploration: 

##### Testing and exploring the dataset before having a defined and modular implementation

In [5]:
amazon_titles = read_json(amazon_file_path)

print(amazon_titles[:5])
print("Length:", len(amazon_titles))

['Amazon Singapore Celebrates Prime Day with Prime Day Carnival', 'In 50 Words: Amazon to invest extra $6.5b in India by 2030, total at $26b - Tech in Asia | Flipboard', 'Create product videos for your Amazon listings', 'MASSACHUSETTS KIDS REJECT WOKE PROGRAMMING! AMAZON PULLS A THEIRTUBE! "SMART" TOILET ANALPRINTING?!', 'Satellite tv for pc Web. the way it might turn out to be Elon Musk, Tata and Amazon vs Reliance']
Length: 8369


In [6]:
unique_amazon_titles = list(dict.fromkeys(amazon_titles))
print(unique_amazon_titles[:5])
print(len(unique_amazon_titles))

['Amazon Singapore Celebrates Prime Day with Prime Day Carnival', 'In 50 Words: Amazon to invest extra $6.5b in India by 2030, total at $26b - Tech in Asia | Flipboard', 'Create product videos for your Amazon listings', 'MASSACHUSETTS KIDS REJECT WOKE PROGRAMMING! AMAZON PULLS A THEIRTUBE! "SMART" TOILET ANALPRINTING?!', 'Satellite tv for pc Web. the way it might turn out to be Elon Musk, Tata and Amazon vs Reliance']
7257


### Tokenizing the titles and also removing the punctuations for better score output. 

Clean the Titles from punctuation to increase the similarity score

In [7]:
import string

def clean_titles(title):
    valid_words = []
    for word in title.split():
        clean_word = word.strip(string.punctuation)
        if clean_word in model:
            valid_words.append(clean_word)
    return valid_words

test_title = unique_amazon_titles[1]
cleaned_test_title = clean_titles(test_title)
print("Test Title: ", test_title)
print("Cleaned Title: ", cleaned_test_title)

Test Title:  In 50 Words: Amazon to invest extra $6.5b in India by 2030, total at $26b - Tech in Asia | Flipboard
Cleaned Title:  ['In', 'Words', 'Amazon', 'invest', 'extra', 'in', 'India', 'by', 'total', 'at', 'Tech', 'in', 'Asia', 'Flipboard']


### Baseline Exploration: Testing the Naive Approach vs improved approach
##### Testing the model with just str.split() vs the clean_titles method to see the similarity score.

In [8]:
title_a = unique_amazon_titles[0]
title_b = unique_amazon_titles[1]
print("Title A: ", title_a)
print("Title B: ", title_b)

score = model.n_similarity(title_a.split(), title_b.split())
print("\nSimilarity Score: ", score)

punctuation_removed_score = model.n_similarity(clean_titles(title_a), clean_titles(title_b))
print("Similarity Score when punctuations are removed: ", punctuation_removed_score)

Title A:  Amazon Singapore Celebrates Prime Day with Prime Day Carnival
Title B:  In 50 Words: Amazon to invest extra $6.5b in India by 2030, total at $26b - Tech in Asia | Flipboard

Similarity Score:  0.38621038
Similarity Score when punctuations are removed:  0.41973343


Implement and test the similarity score for the Amazon dataset

In [9]:
import pandas as pd

similarity = []
for title in unique_amazon_titles:
    if title == test_title:
        continue

    candidate = clean_titles(title)
    if not candidate:
        continue
    score = model.n_similarity(cleaned_test_title, candidate)
    similarity.append({"text": title, "score": score})

df = pd.DataFrame(similarity)
df.head()

,text,score
0,Amazon Singapore Celebrates Prime Day with Pri...,0.419733
1,Create product videos for your Amazon listings,0.497019
2,MASSACHUSETTS KIDS REJECT WOKE PROGRAMMING! AM...,0.112022
3,Satellite tv for pc Web. the way it might turn...,0.632886
4,In 50 Words: Amazon to invest extra $6.5b in I...,0.894351


In [10]:
df_sorted_amazon = df.sort_values(by="score", ascending=False).reset_index(drop=True)
top_50_amazon = df_sorted_amazon.head(50)
top_50_amazon.head()

,text,score
0,In 50 Words: Amazon to invest extra $6.5b in I...,0.894351
1,In 50 Words: Amazon to invest $15b more in India,0.836391
2,Amazon plans to invest another $15 billion in ...,0.824361
3,In 50 Words: Amazon excludes rival Temu from a...,0.803378
4,Amazon commits to invest $26B in India through...,0.795582


In [11]:
print("Title compared with: ", test_title, "\n\n")
    
for i, row in top_50_amazon.iterrows():
    print(f"{i+1}) [{row['score']:.4f}] Title: {row['text']}")

Title compared with:  In 50 Words: Amazon to invest extra $6.5b in India by 2030, total at $26b - Tech in Asia | Flipboard 


1) [0.8944] Title: In 50 Words: Amazon to invest extra $6.5b in India by 2030, total at $26b
2) [0.8364] Title: In 50 Words: Amazon to invest $15b more in India
3) [0.8244] Title: Amazon plans to invest another $15 billion in India by 2030 | Flipboard
4) [0.8034] Title: In 50 Words: Amazon excludes rival Temu from algorithm over fair pricing concerns - Tech in Asia | Flipboard
5) [0.7956] Title: Amazon commits to invest $26B in India through 2030 - SiliconANGLE | Flipboard
6) [0.7503] Title: Amazon plans to invest another $15 billion in India by 2030
7) [0.7358] Title: Amazon to more than double its investment in India to $26bn by 2030
8) [0.7341] Title: Amazon Prime Lite launched in India: Benefits of new plan and how to sign up | Flipboard
9) [0.7340] Title: Amazon to invest $15 billion in India by 2030 - The Hindu BusinessLine
10) [0.7298] Title: Amazon is te

# Modularized Approach:

Creating a function to make it modularized and working for any title and data set!

In [12]:
def similar_title(news_title: str, news_titles: list):
    tokenized_news_title = clean_titles(news_title)
    result = []
    for candidate_news in news_titles:
        if news_title == candidate_news:
            continue
        tokenized_candidate_title = clean_titles(candidate_news)
        if not tokenized_candidate_title:
            continue
        
        similarity_score = model.n_similarity(tokenized_news_title, tokenized_candidate_title)

        result.append({"Title": candidate_news, "Score": similarity_score})

    return result

In [13]:
def top_50_similar_news(input_news):
    data_frame = pd.DataFrame(input_news)
    data_frame = data_frame.sort_values(by="Score", ascending=False).reset_index(drop=True).head(50)

    return data_frame

In [14]:
def display_top_50_news(input_news):
    for i, row in input_news.iterrows():
        print(f"{i+1}) [{row['Score']:.4f}] Title: {row['Title']}")

In [15]:
amazon_focus_title = unique_amazon_titles[0]
amazon_news = similar_title(amazon_focus_title, unique_amazon_titles)
amazon_news[:5]

[{'Title': 'In 50 Words: Amazon to invest extra $6.5b in India by 2030, total at $26b - Tech in Asia | Flipboard',
  'Score': 0.41973343},
 {'Title': 'Create product videos for your Amazon listings',
  'Score': 0.34567124},
 {'Title': 'MASSACHUSETTS KIDS REJECT WOKE PROGRAMMING! AMAZON PULLS A THEIRTUBE! "SMART" TOILET ANALPRINTING?!',
  'Score': 0.23101375},
 {'Title': 'Satellite tv for pc Web. the way it might turn out to be Elon Musk, Tata and Amazon vs Reliance',
  'Score': 0.3423766},
 {'Title': 'In 50 Words: Amazon to invest extra $6.5b in India by 2030, total at $26b',
  'Score': 0.43131614}]

In [16]:
amazon_news_df = top_50_similar_news(amazon_news)

print("Title compared with: ", amazon_focus_title, "\n\n")
display_top_50_news(amazon_news_df)

Title compared with:  Amazon Singapore Celebrates Prime Day with Prime Day Carnival 


1) [0.7903] Title: Amazon Prime Day 2023 Announced
2) [0.7888] Title: 32 of the Best Prime Day Deals 2023 to Score Now Ahead of Amazon Prime Day | Vogue
3) [0.7782] Title: 31 of the Best Early Prime Day Day Deals From Amazon for 2023
4) [0.7742] Title: Amazon Announces the Dates for Its Annual Prime Day Event
5) [0.7741] Title: Amazon Prime Day Returns on 11 and 12 July with Thousands of Real Deals and Exclusive Launches in Singapore
6) [0.7580] Title: Amazon Prime Day Sale: Starts July 11-12 | Early Deals Now Live
7) [0.7544] Title: Target Circle Week: 7-Day Sales Event in July Competes With Amazon Prime Day - CNET
8) [0.7538] Title: Amazon: Prime Day 2023 event is July 11-12 PLUS early Prime Day deals are live now!
9) [0.7523] Title: Amazon Prime Day 2023 Dates Announces: July 11-12 - Variety
10) [0.7515] Title: Amazon Prime Day 2023 Announced: 48-Hour Sale Kicks Off July 11
11) [0.7472] Title: Wal

TESLA DATASET

In [17]:
tesla_titles = read_json(tesla_file_path)
print(tesla_titles[:5])
print("Length:", len(tesla_titles))

['Tesla Update: Confirmation Either Way', 'China’s ‘Tesla Killer’ Stumbles as EV Price War Takes Toll - WSJ', 'China’s ‘Tesla Killer’ Stumbles as EV Price War Takes Toll', 'Automobile rivals embrace Tesla EV chargers - The Hindu', 'Nicola Tesla / Ocean Gate Connection : Fuel-less Generators Scuttled (2/2)']
Length: 4039


In [18]:
unique_tesla_titles = list(dict.fromkeys(tesla_titles))
print(unique_tesla_titles[:5])
print(len(unique_tesla_titles))

['Tesla Update: Confirmation Either Way', 'China’s ‘Tesla Killer’ Stumbles as EV Price War Takes Toll - WSJ', 'China’s ‘Tesla Killer’ Stumbles as EV Price War Takes Toll', 'Automobile rivals embrace Tesla EV chargers - The Hindu', 'Nicola Tesla / Ocean Gate Connection : Fuel-less Generators Scuttled (2/2)']
3683


In [19]:
tesla_focus_title = unique_tesla_titles[1]
tesla_news = similar_title(tesla_focus_title, unique_tesla_titles)
tesla_news[:5]

[{'Title': 'Tesla Update: Confirmation Either Way', 'Score': 0.47993314},
 {'Title': 'China’s ‘Tesla Killer’ Stumbles as EV Price War Takes Toll',
  'Score': 0.9724037},
 {'Title': 'Automobile rivals embrace Tesla EV chargers - The Hindu',
  'Score': 0.3578694},
 {'Title': 'Nicola Tesla / Ocean Gate Connection : Fuel-less Generators Scuttled (2/2)',
  'Score': 0.47615436},
 {'Title': "Tesla's Chief Designer Takes Cybertruck For A Spin, Gives Sneak Peek Into 'Spacious Interiors,' Other Features - Tesla (NASDAQ:TSLA) - Benzinga",
  'Score': 0.65833485}]

In [20]:
tesla_news_df = top_50_similar_news(tesla_news)

print("Title compared with: ", tesla_focus_title, "\n\n")
display_top_50_news(tesla_news_df)

Title compared with:  China’s ‘Tesla Killer’ Stumbles as EV Price War Takes Toll - WSJ 


1) [0.9724] Title: China’s ‘Tesla Killer’ Stumbles as EV Price War Takes Toll
2) [0.7710] Title: Tesla Analyst Drops Bullish Stance As Near-Term Storm Takes Shape, But Sees Elon Musk-Led Company As Long-Term EV Winner - Tesla (NASDAQ:TSLA) - Benzinga
3) [0.7636] Title: Dow Jones Wavers, Tech Takes Lead As Powell Answers Bank Run Question; Tesla Rises As AI Investor Day Looms | Investor's Business Daily
4) [0.7534] Title: Tesla Pulls Dramatic Turnaround as EV Maker Powers Back to Exclusive Ranking | Thestreet | kilgorenewsherald.com
5) [0.7406] Title: Dow Jones Wavers, Tech Takes Lead As Powell Answers Bank Run Question; Tesla, Amazon Rise As AI Investor Day Starts | Investor's Business Daily
6) [0.7369] Title: Stock Market Trims Gains As Small Caps Jump. Tesla Goes On Amazing Run And Apple Gets Bullish Call. | Investor's Business Daily
7) [0.7324] Title: Tesla Pulls Dramatic Turnaround as EV Maker